In [1]:
from pathlib import Path

import matplotlib

matplotlib.use("Agg")

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

In [2]:
def resolve_archive_dirs():
    cwd = Path.cwd()
    candidates = [cwd, cwd / "lab3"]
    for root in candidates:
        csv_path = root / "archive" / "chinese_mnist.csv"
        img_dir = root / "archive" / "data" / "data"
        if csv_path.is_file() and img_dir.is_dir():
            return root, csv_path, img_dir
    raise FileNotFoundError(
        "未找到 archive（chinese_mnist.csv 与 data/data/）。请将 notebook 工作目录设为 lab3。"
    )


LAB_ROOT, CSV_PATH, IMG_DIR = resolve_archive_dirs()
print("LAB_ROOT:", LAB_ROOT)
print("CSV:", CSV_PATH)
print("图像目录:", IMG_DIR)

df = pd.read_csv(CSV_PATH)
df["path"] = df.apply(
    lambda r: IMG_DIR / f"input_{int(r.suite_id)}_{int(r.sample_id)}_{int(r.code)}.jpg",
    axis=1,
)

n_rows = len(df)
print("\n标注行数:", n_rows)
print("列名:", list(df.columns))

label_col = "code"
n_classes = df[label_col].nunique()
print("类别数:", n_classes)
display(df[label_col].value_counts().sort_index())

LAB_ROOT: d:\Cursor\UNSW-COMP-9517\lab3
CSV: d:\Cursor\UNSW-COMP-9517\lab3\archive\chinese_mnist.csv
图像目录: d:\Cursor\UNSW-COMP-9517\lab3\archive\data\data

标注行数: 15000
列名: ['suite_id', 'sample_id', 'code', 'value', 'character', 'path']
类别数: 15


code
1     1000
2     1000
3     1000
4     1000
5     1000
6     1000
7     1000
8     1000
9     1000
10    1000
11    1000
12    1000
13    1000
14    1000
15    1000
Name: count, dtype: int64

In [3]:
sample_path = df.loc[0, "path"]
sample = cv2.imread(str(sample_path), cv2.IMREAD_GRAYSCALE)
assert sample is not None, f"无法读取 {sample_path}"
print("示例路径:", sample_path)
print("单张尺寸 (H, W):", sample.shape)
print("dtype:", sample.dtype)

示例路径: d:\Cursor\UNSW-COMP-9517\lab3\archive\data\data\input_1_1_10.jpg
单张尺寸 (H, W): (64, 64)
dtype: uint8


In [4]:
fig, axes = plt.subplots(3, 5, figsize=(12, 7))
axes = axes.ravel()
codes = sorted(df[label_col].unique())
for ax, c in zip(axes, codes):
    row = df[df[label_col] == c].iloc[0]
    img = cv2.imread(str(row["path"]), cv2.IMREAD_GRAYSCALE)
    ax.imshow(img, cmap="gray")
    ch = row.get("character", "")
    ax.set_title(f"code={c} {ch}")
    ax.axis("off")
plt.suptitle("每类 1 张示例", fontsize=14)
plt.tight_layout()
plt.show()

C:\Users\hrole\AppData\Local\Temp\ipykernel_17276\2344144808.py:12: UserWarning: Glyph 38646 (\N{CJK UNIFIED IDEOGRAPH-96F6}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\hrole\AppData\Local\Temp\ipykernel_17276\2344144808.py:12: UserWarning: Glyph 19968 (\N{CJK UNIFIED IDEOGRAPH-4E00}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\hrole\AppData\Local\Temp\ipykernel_17276\2344144808.py:12: UserWarning: Glyph 20108 (\N{CJK UNIFIED IDEOGRAPH-4E8C}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\hrole\AppData\Local\Temp\ipykernel_17276\2344144808.py:12: UserWarning: Glyph 19977 (\N{CJK UNIFIED IDEOGRAPH-4E09}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\hrole\AppData\Local\Temp\ipykernel_17276\2344144808.py:12: UserWarning: Glyph 22235 (\N{CJK UNIFIED IDEOGRAPH-56DB}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\hrole\AppData\Local\Temp\ipykernel_17276\2344144808.py:12: UserWarning: Glyph 20116 (\

In [5]:
X_list, y_list = [], []
for _, row in df.iterrows():
    img = cv2.imread(str(row["path"]), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(row["path"])
    X_list.append(img.reshape(-1))
    y_list.append(int(row[label_col]))

X = np.asarray(X_list, dtype=np.float32)
y = np.asarray(y_list)
print("X.shape:", X.shape)
print("y.shape:", y.shape)
LABELS = np.sort(np.unique(y))

X.shape: (15000, 4096)
y.shape: (15000,)


In [6]:
RANDOM_STATE = 42

X_pool, X_test, y_pool, y_test = train_test_split(
    X,
    y,
    test_size=1000,
    stratify=y,
    random_state=RANDOM_STATE,
)


def stratified_train_subset(X_p, y_p, n_train, random_state):
    sss = StratifiedShuffleSplit(
        n_splits=1, train_size=n_train, random_state=random_state
    )
    for train_idx, _ in sss.split(X_p, y_p):
        return X_p[train_idx], y_p[train_idx]
    raise RuntimeError("StratifiedShuffleSplit 未产生划分")


X_train_5k, y_train_5k = stratified_train_subset(
    X_pool, y_pool, 5000, random_state=RANDOM_STATE
)
X_train_10k, y_train_10k = stratified_train_subset(
    X_pool, y_pool, 10000, random_state=RANDOM_STATE + 1
)

print("测试集:", X_test.shape[0])
print("训练 5k:", X_train_5k.shape[0])
print("训练 10k:", X_train_10k.shape[0])
print("训练与测试交集（特征+标签行是否同一对象）: 应为 0 行")
# 行级唯一性：用原始索引不可行，此处检查规模与分层计数
print("\n测试集每类样本数:\n", pd.Series(y_test).value_counts().sort_index())
print("\n训练 5k 每类样本数:\n", pd.Series(y_train_5k).value_counts().sort_index())

测试集: 1000
训练 5k: 5000
训练 10k: 10000
训练与测试交集（特征+标签行是否同一对象）: 应为 0 行

测试集每类样本数:
 1     66
2     67
3     67
4     67
5     67
6     66
7     67
8     67
9     67
10    66
11    67
12    66
13    67
14    66
15    67
Name: count, dtype: int64

训练 5k 每类样本数:
 1     334
2     333
3     333
4     333
5     333
6     334
7     333
8     333
9     333
10    334
11    333
12    334
13    333
14    334
15    333
Name: count, dtype: int64


In [7]:
def run_and_report(X_tr, y_tr, X_te, y_te, experiment_title):
    classifiers = {
        "KNN (k=3)": KNeighborsClassifier(n_neighbors=3),
        "Decision Tree (default)": DecisionTreeClassifier(),
        "SGD (max_iter=250)": SGDClassifier(max_iter=250),
    }

    print("\n" + "=" * 72)
    print(experiment_title)
    print("训练样本:", X_tr.shape[0], " 测试样本:", X_te.shape[0])
    print("=" * 72)

    for name, clf in classifiers.items():
        clf.fit(X_tr, y_tr)
        y_pred = clf.predict(X_te)

        acc = accuracy_score(y_te, y_pred)
        prec = precision_score(y_te, y_pred, average="macro", zero_division=0)
        rec = recall_score(y_te, y_pred, average="macro", zero_division=0)
        f1 = f1_score(y_te, y_pred, average="macro", zero_division=0)
        cm = confusion_matrix(y_te, y_pred, labels=LABELS)

        print(f"\n--- {name} ---")
        print(f"Accuracy : {acc:.4f}")
        print(f"Precision (macro): {prec:.4f}")
        print(f"Recall    (macro): {rec:.4f}")
        print(f"F1-score  (macro): {f1:.4f}")

        fig, ax = plt.subplots(figsize=(8, 6))
        im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
        ax.figure.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        ax.set(
            xticks=np.arange(cm.shape[1]),
            yticks=np.arange(cm.shape[0]),
            xticklabels=LABELS,
            yticklabels=LABELS,
            ylabel="True code",
            xlabel="Predicted code",
            title=f"{name} — Confusion matrix",
        )
        plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
        thresh = cm.max() / 2.0 if cm.max() > 0 else 0
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                ax.text(
                    j,
                    i,
                    format(cm[i, j], "d"),
                    ha="center",
                    va="center",
                    color="white" if cm[i, j] > thresh else "black",
                    fontsize=7,
                )
        fig.tight_layout()
        plt.show()


run_and_report(
    X_train_5k,
    y_train_5k,
    X_test,
    y_test,
    "实验 A：5,000 训练 / 1,000 测试（分层，同一测试集）",
)

run_and_report(
    X_train_10k,
    y_train_10k,
    X_test,
    y_test,
    "实验 B：10,000 训练 / 1,000 测试（分层，同一测试集）",
)


实验 A：5,000 训练 / 1,000 测试（分层，同一测试集）
训练样本: 5000  测试样本: 1000

--- KNN (k=3) ---
Accuracy : 0.3510
Precision (macro): 0.5631
Recall    (macro): 0.3504
F1-score  (macro): 0.3582


C:\Users\hrole\AppData\Local\Temp\ipykernel_17276\2814614259.py:55: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



--- Decision Tree (default) ---
Accuracy : 0.2800
Precision (macro): 0.2752
Recall    (macro): 0.2797
F1-score  (macro): 0.2758


C:\Users\hrole\AppData\Local\Temp\ipykernel_17276\2814614259.py:55: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



--- SGD (max_iter=250) ---
Accuracy : 0.3020
Precision (macro): 0.3278
Recall    (macro): 0.3019
F1-score  (macro): 0.3096

实验 B：10,000 训练 / 1,000 测试（分层，同一测试集）
训练样本: 10000  测试样本: 1000


...

C:\Users\hrole\AppData\Local\Temp\ipykernel_17276\2814614259.py:55: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



--- Decision Tree (default) ---
Accuracy : 0.3110
Precision (macro): 0.3078
Recall    (macro): 0.3106
F1-score  (macro): 0.3083


C:\Users\hrole\AppData\Local\Temp\ipykernel_17276\2814614259.py:55: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



--- SGD (max_iter=250) ---
Accuracy : 0.3190
Precision (macro): 0.3366
Recall    (macro): 0.3190
F1-score  (macro): 0.3181


C:\Users\hrole\AppData\Local\Temp\ipykernel_17276\2814614259.py:55: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
